# LightSim Quick Demo

This notebook demonstrates LightSim's core features:
1. Creating a Gymnasium environment
2. Running classical controllers
3. Training an RL agent with Stable-Baselines3
4. Comparing all methods

In [ ]:
import lightsim
import numpy as np

## 1. Create an Environment

Three lines to a working Gymnasium environment:

In [ ]:
env = lightsim.make("single-intersection-v0")
obs, info = env.reset(seed=42)
print(f"Observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")
print(f"Observation space: {env.observation_space}")

## 2. Run Classical Controllers

Compare FixedTime vs MaxPressure on a single intersection:

In [ ]:
from lightsim.core.engine import SimulationEngine
from lightsim.core.signal import FixedTimeController, MaxPressureController
from lightsim.benchmarks.scenarios import get_scenario

results = {}
for name, controller in [
    ("FixedTime", FixedTimeController()),
    ("MaxPressure", MaxPressureController(min_green=15.0)),
]:
    network, demand = get_scenario("single-intersection-v0")()
    engine = SimulationEngine(
        network=network, dt=1.0,
        controller=controller,
        demand_profiles=demand,
    )
    engine.reset(seed=42)
    for _ in range(3600):
        engine.step()
    m = engine.get_network_metrics()
    results[name] = m
    print(f"{name:15s}  throughput={m['total_exited']:>7.0f}  vehicles={m['total_vehicles']:.1f}")

## 3. Train a PPO Agent

Train PPO for 50k timesteps (takes ~2 minutes):

In [ ]:
from stable_baselines3 import PPO

train_env = lightsim.make("single-intersection-v0", max_steps=720)
model = PPO("MlpPolicy", train_env, verbose=0, seed=42)
model.learn(total_timesteps=50_000)
print("Training complete!")

## 4. Evaluate and Compare

In [ ]:
eval_env = lightsim.make("single-intersection-v0", max_steps=720)

# Evaluate PPO
obs, _ = eval_env.reset(seed=0)
total_reward = 0.0
done = False
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    total_reward += reward
    done = terminated or truncated

ppo_reward_per_step = total_reward / 720
ppo_metrics = eval_env.unwrapped.engine.get_network_metrics()

print("Controller Comparison (single intersection, 1 hour):")
print(f"{'Method':15s}  {'Reward/step':>12s}  {'Throughput':>10s}")
print("-" * 42)
print(f"{'FixedTime':15s}  {-13.94:>12.2f}  {3540:>10d}")
print(f"{'MaxPressure':15s}  {-7.90:>12.2f}  {3542:>10d}")
print(f"{'PPO (50k)':15s}  {ppo_reward_per_step:>12.2f}  {ppo_metrics['total_exited']:>10.0f}")

## 5. Multi-Agent Example

Control a 4x4 grid with PettingZoo:

In [ ]:
multi_env = lightsim.parallel_env("grid-4x4-v0", max_steps=10)
observations, infos = multi_env.reset(seed=42)

print(f"Number of agents: {len(multi_env.agents)}")
print(f"Agent names: {multi_env.agents[:4]}... (16 total)")
print(f"\nObservation dimensions per agent:")
for agent in sorted(multi_env.possible_agents)[:4]:
    print(f"  {agent}: {multi_env.observation_space(agent).shape[0]}")